# Phase 7: Mediated Risk Effects + Joint PAF

This notebook demonstrates the Phase 7 model build-up, which adds:

- **MediatedRiskEffect** (log-linear): BMI on IS/MI and HF targets, categorical SBP on HF
- **NonLogLinearMediatedRiskEffect**: SBP, LDL-C, FPG on IS/MI targets
- **JointPAF**: population-attributable fractions computed from a dedicated PAF-calculation simulation and cached in the artifact

The mediation math adjusts each risk's relative risk to remove the portion of its effect that operates through downstream mediators (e.g., BMI's effect on MI is partially mediated by SBP, LDL-C, and FPG).

> **Note:** BMI relative risk data for IS/MI targets is currently **stub data** (RR=1.0, i.e. no direct effect). The GBD 2023 artifact was missing these entries. This needs investigation — see `memory/project_bmi_is_mi_investigation.md`.

In [1]:
from vivarium import InteractiveContext
import pandas as pd
import numpy as np

yaml_path = '../src/vivarium_nih_us_cvd/model_specifications/nih_us_cvd_phase7.yaml'
sim = InteractiveContext(yaml_path, setup=False)
sim.configuration.update({'population': {'population_size': 1_000}})
sim.setup()
print('Setup complete.')

2026-04-12 07:59:29.307 | INFO     | simulation_1-artifact_manager:80 - Running simulation from artifact located at /home/abie/vivarium_nih_us_cvd/src/vivarium_nih_us_cvd/artifacts/united_states_of_america.hdf.


2026-04-12 07:59:29.308 | INFO     | simulation_1-artifact_manager:81 - Artifact base filter terms are [].


2026-04-12 07:59:29.308 | INFO     | simulation_1-artifact_manager:82 - Artifact additional filter terms are None.


2026-04-12 08:04:52.113 | WARNING  | simulation_1-values_manager:406 - Conflicting information for acute_ischemic_stroke.incidence_rate.paf. Ignoring 'required_resources' since the `modifier` is of type <class 'vivarium.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-04-12 08:04:52.115 | WARNING  | simulation_1-values_manager:406 - Conflicting information for acute_myocardial_infarction.incidence_rate.paf. Ignoring 'required_resources' since the `modifier` is of type <class 'vivarium.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


...

2026-04-12 08:04:53.026 | WARNING  | simulation_1-lookup_table_manager:85 - Component 'risk_factor.ldlc_medication_adherence' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.


2026-04-12 08:04:53.028 | INFO     | simulation_1-results_context:131 - The following stratifications are registered but not used by any observers: 
['event_year']


2026-04-12 08:04:55.864 | WARNING  | simulation_1-population_manager:747 - The 'risk_factor.outreach.exposure_parameters.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'risk_factor.outreach.exposure_parameters.paf'.


2026-04-12 08:04:55.869 | WARNING  | simulation_1-population_manager:747 - The 'risk_factor.polypill.exposure_parameters.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'risk_factor.polypill.exposure_parameters.paf'.


Setup complete.


## Component inventory

Phase 7 adds 20 mediated risk effects (vs. 3 simple effects in Phase 6) plus the JointPAF component.

In [2]:
components = sim._component_manager.list_components()

log_linear = [c for c in components if c.startswith('risk_effect.')]
non_ll = [c for c in components if c.startswith('non_log_linear_risk_effect.')]
joint_paf = [c for c in components if 'joint_paf' in c]

print(f'Log-linear mediated effects: {len(log_linear)}')
print(f'Non-log-linear mediated effects: {len(non_ll)}')
print(f'JointPAF components: {len(joint_paf)}')
print()
print('All risk effect components:')
for c in sorted(log_linear + non_ll):
    print(f'  {c}')

Log-linear mediated effects: 8
Non-log-linear mediated effects: 12
JointPAF components: 1

All risk effect components:
  non_log_linear_risk_effect.high_fasting_plasma_glucose_on_cause.acute_ischemic_stroke.incidence_rate
  non_log_linear_risk_effect.high_fasting_plasma_glucose_on_cause.acute_myocardial_infarction.incidence_rate
  non_log_linear_risk_effect.high_fasting_plasma_glucose_on_cause.chronic_ischemic_stroke_to_acute_ischemic_stroke.transition_rate
  non_log_linear_risk_effect.high_fasting_plasma_glucose_on_cause.post_myocardial_infarction_to_acute_myocardial_infarction.transition_rate
  non_log_linear_risk_effect.high_ldl_cholesterol_on_cause.acute_ischemic_stroke.incidence_rate
  non_log_linear_risk_effect.high_ldl_cholesterol_on_cause.acute_myocardial_infarction.incidence_rate
  non_log_linear_risk_effect.high_ldl_cholesterol_on_cause.chronic_ischemic_stroke_to_acute_ischemic_stroke.transition_rate
  non_log_linear_risk_effect.high_ldl_cholesterol_on_cause.post_myocardial_i

## Run a short simulation

Step for ~6 months (6 x 28-day steps) and inspect disease transitions.

In [3]:
for i in range(6):
    sim.step()
print(f'Simulated through 6 steps (168 days).')

2026-04-12 08:04:56.644 | INFO     | simulation_1 - vivarium.framework.engine:280 - 2024-01-01 00:00:00


2026-04-12 08:04:57.789 | WARNING  | simulation_1-population_manager:747 - The 'affected_unmodeled.cause_specific_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'affected_unmodeled.cause_specific_mortality_rate.paf'.


2026-04-12 08:04:57.796 | WARNING  | simulation_1-population_manager:747 - The 'acute_ischemic_stroke.excess_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'acute_ischemic_stroke.excess_mortality_rate.paf'.


2026-04-12 08:04:57.812 | WARNING  | simulation_1-population_manager:747 - The 'chronic_ischemic_stroke.excess_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'chronic_ischemic_stroke.excess_mortality_rate.paf'.


2026-04-12 08:04:57.820 | WARNING  | simulation_1-population_manager:747 - The 'acute_myocardial_infarction.excess_mortality_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'acute_myocardial_infarction.excess_mortality_rate.paf'.


...

2026-04-12 08:05:48.905 | WARNING  | simulation_1-population_manager:747 - The 'post_myocardial_infarction_to_acute_myocardial_infarction.transition_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'post_myocardial_infarction_to_acute_myocardial_infarction.transition_rate.paf'.


2026-04-12 08:05:49.375 | WARNING  | simulation_1-population_manager:747 - The 'post_myocardial_infarction_to_heart_failure_from_ischemic_heart_disease.transition_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'post_myocardial_infarction_to_heart_failure_from_ischemic_heart_disease.transition_rate.paf'.


2026-04-12 08:05:49.412 | WARNING  | simulation_1-population_manager:747 - The 'heart_failure_from_ischemic_heart_disease_to_acute_myocardial_infarction_and_heart_failure.transition_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'heart_failure_from_ischemic_heart_disease_to_acute_myocardial_infarction_and_heart_failure.transition_rate.paf'.


2026-04-12 08:05:49.972 | WARNING  | simulation_1-population_manager:747 - The 'risk_factor.lifestyle.exposure_parameters.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'risk_factor.lifestyle.exposure_parameters.paf'.


Simulated through 6 steps (168 days).


In [4]:
pop = sim.get_population([
    'is_alive', 'age', 'sex',
    'ischemic_stroke',
    'ischemic_heart_disease_and_heart_failure',
])

print(f'Population: {len(pop)} simulants, {pop["is_alive"].sum()} alive')
print()
for col in ['ischemic_stroke', 'ischemic_heart_disease_and_heart_failure']:
    print(f'{col}:')
    print(pop[col].value_counts().to_string())
    print()

Population: 1000 simulants, 997 alive

ischemic_stroke:
ischemic_stroke
susceptible_to_ischemic_stroke    975
chronic_ischemic_stroke            25

ischemic_heart_disease_and_heart_failure:
ischemic_heart_disease_and_heart_failure
susceptible_to_ischemic_heart_disease_and_heart_failure    983
heart_failure_from_ischemic_heart_disease                    7
heart_failure_residual                                       7
post_myocardial_infarction                                   3



## Population-Attributable Fractions

The JointPAF component loads precomputed PAFs from the artifact and applies them as modifiers on each target rate's `.paf` pipeline. The PAF = (E[RR] - 1) / E[RR] is computed per age/sex stratum in a dedicated PAF-calculation simulation with 100K simulants.

In [5]:
paf_cols = [
    'acute_ischemic_stroke.incidence_rate.paf',
    'acute_myocardial_infarction.incidence_rate.paf',
    'heart_failure_from_ischemic_heart_disease.incidence_rate.paf',
    'heart_failure_residual.incidence_rate.paf',
]

paf_pop = sim.get_population(paf_cols + ['age', 'sex'])

print('PAF summary by target rate:')
print('=' * 70)
for col in paf_cols:
    vals = paf_pop[col]
    short_name = col.replace('.incidence_rate.paf', '')
    print(f'{short_name:50s}  mean={vals.mean():.3f}  [{vals.min():.3f}, {vals.max():.3f}]')
print()
print('PAF of 0 in the youngest bin is expected (risks have no effect below age 25).')

2026-04-12 08:05:52.653 | WARNING  | simulation_1-population_manager:747 - The 'acute_myocardial_infarction.incidence_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'acute_myocardial_infarction.incidence_rate.paf'.


2026-04-12 08:05:52.665 | WARNING  | simulation_1-population_manager:747 - The 'heart_failure_from_ischemic_heart_disease.incidence_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'heart_failure_from_ischemic_heart_disease.incidence_rate.paf'.


2026-04-12 08:05:52.673 | WARNING  | simulation_1-population_manager:747 - The 'acute_ischemic_stroke.incidence_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'acute_ischemic_stroke.incidence_rate.paf'.


2026-04-12 08:05:52.683 | WARNING  | simulation_1-population_manager:747 - The 'heart_failure_residual.incidence_rate.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'heart_failure_residual.incidence_rate.paf'.


PAF summary by target rate:
acute_ischemic_stroke                               mean=0.570  [0.000, 0.838]
acute_myocardial_infarction                         mean=0.528  [0.000, 0.787]
heart_failure_from_ischemic_heart_disease           mean=0.258  [0.047, 0.431]
heart_failure_residual                              mean=0.258  [0.047, 0.431]

PAF of 0 in the youngest bin is expected (risks have no effect below age 25).


In [6]:
# PAF by age group for acute IS
paf_pop['age_group'] = pd.cut(
    paf_pop['age'],
    bins=[0, 25, 35, 45, 55, 65, 75, 85, 130],
    labels=['<25', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+'],
    right=False,
)

paf_by_age = paf_pop.groupby(['age_group', 'sex'])[
    'acute_ischemic_stroke.incidence_rate.paf'
].mean().unstack('sex')

print('IS incidence PAF by age and sex:')
print(paf_by_age.round(3).to_string())

IS incidence PAF by age and sex:
sex        Female   Male
age_group               
<25         0.000  0.000
25-34       0.687  0.837
35-44       0.740  0.823
45-54       0.768  0.820
55-64       0.785  0.798
65-74       0.783  0.750
75-84       0.783  0.740
85+         0.764  0.709


/tmp/ipykernel_293579/4246676292.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  paf_by_age = paf_pop.groupby(['age_group', 'sex'])[


## How mediation works

For a risk like BMI affecting MI, the effect is mediated by SBP, LDL-C, and FPG. The mediated target modifier computes:

```
adjusted_rate = base_rate * unadjusted_RR / scaling_factor
```

where `scaling_factor` accounts for the portion of BMI's effect operating through each mediator:

```
for each mediator:
    mf = mediation_factor(risk, mediator, target)
    delta = log(mf * (RR_risk - 1) + 1) / log(RR_mediator)
    scaling_factor *= RR_mediator^delta
```

For heart failure targets, precomputed delta values from the artifact are used instead of the mediation factor formula.

The `MEDIATOR_NAMES` dict defines which risks mediate which targets:

In [7]:
from vivarium_nih_us_cvd.components.effects import MEDIATOR_NAMES

for risk, targets in MEDIATOR_NAMES.items():
    print(f'{risk}:')
    for target, mediators in targets.items():
        print(f'  {target} -> mediators: {mediators}')
    print()

high_body_mass_index_in_adults:
  acute_ischemic_stroke -> mediators: ['high_systolic_blood_pressure', 'high_ldl_cholesterol', 'high_fasting_plasma_glucose']
  chronic_ischemic_stroke_to_acute_ischemic_stroke -> mediators: ['high_systolic_blood_pressure', 'high_ldl_cholesterol', 'high_fasting_plasma_glucose']
  acute_myocardial_infarction -> mediators: ['high_systolic_blood_pressure', 'high_ldl_cholesterol', 'high_fasting_plasma_glucose']
  post_myocardial_infarction_to_acute_myocardial_infarction -> mediators: ['high_systolic_blood_pressure', 'high_ldl_cholesterol', 'high_fasting_plasma_glucose']
  heart_failure_from_ischemic_heart_disease -> mediators: ['categorical_high_systolic_blood_pressure']
  heart_failure_residual -> mediators: ['categorical_high_systolic_blood_pressure']

high_fasting_plasma_glucose:
  acute_ischemic_stroke -> mediators: ['high_ldl_cholesterol']
  chronic_ischemic_stroke_to_acute_ischemic_stroke -> mediators: ['high_ldl_cholesterol']
  acute_myocardial_infarc

## Observer results

Phase 7 inherits all observers from Phase 6: mortality, disability, disease, healthcare visits, medication, lifestyle.

In [8]:
results = sim.get_results()
print(f'Total result measures: {len(results)}')
print()
for key in sorted(results.keys()):
    df = results[key]
    print(f'  {key}: {df.shape}')

Total result measures: 39

  deaths: (512, 8)
  healthcare_visits_background: (64, 4)
  healthcare_visits_emergency: (64, 4)
  healthcare_visits_missed: (64, 4)
  healthcare_visits_none: (64, 4)
  healthcare_visits_scheduled: (64, 4)
  ldlc_medication_high_intensity_person_time: (64, 4)
  ldlc_medication_high_with_eze_person_time: (64, 4)
  ldlc_medication_low_intensity_person_time: (64, 4)
  ldlc_medication_low_med_with_eze_person_time: (64, 4)
  ldlc_medication_medium_intensity_person_time: (64, 4)
  ldlc_medication_no_treatment_person_time: (64, 4)
  lifestyle_cat1_person_time: (64, 4)
  lifestyle_cat2_person_time: (64, 4)
  outreach_cat1_person_time: (64, 4)
  outreach_cat2_person_time: (64, 4)
  person_time_ischemic_heart_disease_and_heart_failure: (384, 8)
  person_time_ischemic_stroke: (192, 8)
  polypill_cat1_person_time: (64, 4)
  polypill_cat2_person_time: (64, 4)
  sbp_medication_no_treatment_person_time: (64, 4)
  sbp_medication_one_drug_half_dose_efficacy_person_time: (64,

## Known limitations

- **BMI -> IS/MI relative risks are stubs (RR=1.0)**. The GBD 2023 artifact did not include BMI relative risk data for ischemic stroke or myocardial infarction targets. Stub data has been added so the model structure is complete, but these need to be replaced with real data before production runs. The model structure (mediation by SBP, LDL-C, FPG) matches the GBD 2020 specification.

- **PAFs need to be recomputed** after the BMI->IS/MI RR data is updated.